# Bi-Encoder Matching Analysis
Threshold sweep analysis, precision-recall tradeoff, and distillation comparison.

In [ ]:
import sys
sys.path.insert(0, '..')
import json
import matplotlib.pyplot as plt
from pathlib import Path

matching_path = Path('../results/matching_results.json')
if matching_path.exists():
    with open(matching_path) as f:
        results = json.load(f)
    cfg_result = results['at_configured_threshold']
    print(f"At threshold {cfg_result['threshold']}:")
    print(f"  Precision: {cfg_result['precision']}")
    print(f"  Recall:    {cfg_result['recall']}")
    print(f"  F1:        {cfg_result['f1']}")
else:
    print('Run matching stage first.')

In [ ]:
# Threshold sweep plot
if matching_path.exists():
    sweep = results['threshold_sweep']
    thresholds = [s['threshold'] for s in sweep]
    precisions = [s['precision'] for s in sweep]
    recalls = [s['recall'] for s in sweep]
    f1s = [s['f1'] for s in sweep]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(thresholds, precisions, label='Precision', marker='o', color='blue')
    ax.plot(thresholds, recalls, label='Recall', marker='o', color='red')
    ax.plot(thresholds, f1s, label='F1', marker='o', color='green', linewidth=2)
    ax.axvline(0.85, color='gray', linestyle='--', alpha=0.7, label='Default threshold (0.85)')
    ax.set_xlabel('Similarity Threshold')
    ax.set_ylabel('Score')
    ax.set_title('Bi-Encoder Matching: Threshold Sweep')
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()

In [ ]:
# Distillation comparison
distill_path = Path('../results/distillation_results.json')
if distill_path.exists():
    with open(distill_path) as f:
        d = json.load(f)
    print('\nDistillation Results:')
    print(f"  Teacher ({d['teacher']['model']}): F1={d['teacher']['f1']}, Size={d['teacher']['size_mb']}MB, {d['teacher']['inference_ms_per_batch']}ms/batch")
    print(f"  Student ({d['student']['model']}): F1={d['student']['f1']}, Size={d['student']['size_mb']}MB, {d['student']['inference_ms_per_batch']}ms/batch")
    print(f"  Compression: {d['compression_ratio']}x  |  Speedup: {d['speedup']}x  |  F1 Retention: {d['f1_retention_pct']}%")

    # Comparison bar chart
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    models = ['Teacher\n(BERT-base)', 'Student\n(DistilBERT)']
    axes[0].bar(models, [d['teacher']['f1'], d['student']['f1']], color=['#4e79a7', '#f28e2b'])
    axes[0].set_title('F1 Score')
    axes[0].set_ylim(0, 1)
    axes[1].bar(models, [d['teacher']['size_mb'], d['student']['size_mb']], color=['#4e79a7', '#f28e2b'])
    axes[1].set_title('Model Size (MB)')
    axes[2].bar(models, [d['teacher']['inference_ms_per_batch'], d['student']['inference_ms_per_batch']], color=['#4e79a7', '#f28e2b'])
    axes[2].set_title('Inference Time (ms/batch)')
    plt.tight_layout()
    plt.show()